# Partie III — RNN, LSTM, GRU et Seq2Seq
### Modélisation de séquences et traduction automatique (corpus réel anglais-français)

**Étudiant·e :** Malak — EMSI, promotion 2025/2026
**Module :** Deep Learning — Partie III (35 points)

Ce notebook couvre, dans l'ordre du cahier des charges :
1. Objectif probabiliste d'un modèle de langage / factorisation par règle de chaîne
2. Perplexité
3. RNN simple, LSTM, GRU (implémentation successive)
4. Comparaison stabilité / performance / mémoire / coût de calcul
5. BPTT et effet du gradient clipping
6. Préparation des données (tokenisation, vocabulaire, tokens spéciaux, padding, mini-lots)
7. Système Seq2Seq encodeur-décodeur
8. Décodage glouton et beam search
9. Évaluation (perplexité, BLEU)
10. Analyse critique et question de synthèse

> **Dataset :** corpus parallèle anglais-français Tatoeba (distribution "Anki", fichier
> `fra.txt`, ~190 000 paires de phrases courtes) — explicitement proposé par le cahier des
> charges ("Tatoeba / fra-eng simplifié"). Le notebook le télécharge automatiquement ; en
> l'absence de connexion il retombe sur un petit corpus de secours embarqué (à ne pas
> utiliser pour les résultats finaux — uniquement pour que le notebook ne plante jamais).
>
> **Sens de traduction choisi : français → anglais.** Les phrases anglaises servent
> également de corpus pour la partie modèle de langage (sections 3-5), ce qui crée un lien
> direct avec le décodeur du Seq2Seq (sections 7-9), qui est lui-même un modèle de langage
> *conditionné* par la phrase source.

**Exécution :** `Exécution > Tout exécuter` dans Colab (GPU recommandé). Le téléchargement
du corpus nécessite une connexion internet.

In [ ]:
# 0. Imports et configuration
import os
import re
import io
import math
import time
import random
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


## 1. Objectif probabiliste d'un modèle de langage

Un modèle de langage estime la probabilité d'une séquence de tokens
$w_1, w_2, \dots, w_T$. Par la **règle de chaîne** des probabilités, cette probabilité
jointe se factorise en un produit de probabilités conditionnelles :

$$P(w_1, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1})$$

C'est exactement ce qu'apprend un RNN/LSTM/GRU entraîné en **next-token prediction** : à
chaque pas de temps $t$, le réseau reçoit l'historique $w_1, \dots, w_{t-1}$ (résumé dans
son état caché $h_{t-1}$) et produit une distribution sur le vocabulaire pour $w_t$.

### Perplexité

La perplexité est l'**exponentielle de la perte d'entropie croisée moyenne** :

$$\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(w_t \mid w_{<t})\right) = e^{\mathcal{L}}$$

**Interprétation** : la perplexité mesure le nombre *effectif* de choix équiprobables que le
modèle "hésite" entre à chaque pas de temps. Une PPL de 1 = prédiction parfaite ; une PPL
égale à la taille du vocabulaire = le modèle ne fait pas mieux qu'une distribution uniforme
(aucun apprentissage utile).

## 2. Préparation des données : tokenisation, vocabulaire, tokens spéciaux, padding

Étapes : téléchargement du corpus réel (avec repli sur un petit corpus embarqué en cas
d'échec réseau), normalisation du texte, tokenisation par mot, construction d'un vocabulaire
avec tokens spéciaux (`<pad>`, `<sos>`, `<eos>`, `<unk>`), filtrage par longueur, puis
numérisation et mise en mini-lots avec padding.

In [ ]:
# 2.1 Corpus de secours (utilisé UNIQUEMENT si le téléchargement échoue -- jamais comme
#     dataset principal pour les résultats finaux du rapport)
FALLBACK_PAIRS = [
    ("I am hungry.", "J'ai faim."),
    ("I am thirsty.", "J'ai soif."),
    ("I am tired.", "Je suis fatigue."),
    ("I am happy.", "Je suis heureux."),
    ("I am sad.", "Je suis triste."),
    ("I am busy.", "Je suis occupe."),
    ("You are kind.", "Tu es gentil."),
    ("You are smart.", "Tu es intelligent."),
    ("He is tall.", "Il est grand."),
    ("She is short.", "Elle est petite."),
    ("We are friends.", "Nous sommes amis."),
    ("They are students.", "Ils sont etudiants."),
    ("This is my book.", "C'est mon livre."),
    ("That is your car.", "C'est ta voiture."),
    ("The weather is nice today.", "Le temps est beau aujourd'hui."),
    ("It is raining outside.", "Il pleut a l'exterieur."),
    ("The sun is shining.", "Le soleil brille."),
    ("I love coffee.", "J'aime le cafe."),
    ("She loves music.", "Elle aime la musique."),
    ("He likes football.", "Il aime le football."),
    ("We like to travel.", "Nous aimons voyager."),
    ("They enjoy reading books.", "Ils aiment lire des livres."),
    ("I live in Casablanca.", "J'habite a Casablanca."),
    ("She lives in Paris.", "Elle habite a Paris."),
    ("He works at a bank.", "Il travaille dans une banque."),
    ("We study computer science.", "Nous etudions l'informatique."),
    ("They study every day.", "Ils etudient tous les jours."),
    ("I want to learn French.", "Je veux apprendre le francais."),
    ("You want to learn English.", "Tu veux apprendre l'anglais."),
    ("He wants to buy a car.", "Il veut acheter une voiture."),
    ("She wants to visit Morocco.", "Elle veut visiter le Maroc."),
    ("We want to eat dinner.", "Nous voulons manger le diner."),
    ("They want to watch a movie.", "Ils veulent regarder un film."),
    ("I can speak three languages.", "Je peux parler trois langues."),
    ("You can play the piano.", "Tu peux jouer du piano."),
    ("He can swim very well.", "Il peut tres bien nager."),
    ("She can run very fast.", "Elle peut courir tres vite."),
    ("We can help you.", "Nous pouvons t'aider."),
    ("They can come tomorrow.", "Ils peuvent venir demain."),
    ("I have a dog.", "J'ai un chien."),
    ("You have a cat.", "Tu as un chat."),
    ("He has a sister.", "Il a une soeur."),
    ("She has two brothers.", "Elle a deux freres."),
    ("We have a meeting today.", "Nous avons une reunion aujourd'hui."),
    ("They have many friends.", "Ils ont beaucoup d'amis."),
    ("I need some help.", "J'ai besoin d'aide."),
    ("You need to rest.", "Tu as besoin de te reposer."),
    ("He needs more time.", "Il a besoin de plus de temps."),
    ("She needs a new phone.", "Elle a besoin d'un nouveau telephone."),
    ("We need to leave now.", "Nous devons partir maintenant."),
    ("They need to study harder.", "Ils doivent etudier plus."),
    ("I went to the market.", "Je suis alle au marche."),
    ("You went to school.", "Tu es alle a l'ecole."),
    ("He went to work early.", "Il est alle au travail tot."),
    ("She went to the cinema.", "Elle est allee au cinema."),
    ("We went to the beach.", "Nous sommes alles a la plage."),
    ("They went to a restaurant.", "Ils sont alles a un restaurant."),
    ("I will call you later.", "Je t'appellerai plus tard."),
    ("You will see the results soon.", "Tu verras les resultats bientot."),
    ("He will finish the project tomorrow.", "Il finira le projet demain."),
    ("She will travel to France next month.", "Elle voyagera en France le mois prochain."),
    ("We will start the meeting now.", "Nous allons commencer la reunion maintenant."),
    ("They will arrive at noon.", "Ils arriveront a midi."),
]
print(f"Corpus de secours : {len(FALLBACK_PAIRS)} paires.")


In [ ]:
# 2.2 Telechargement du corpus reel Tatoeba (Anki fra-eng), avec repli automatique
DATA_URL = "https://www.manythings.org/anki/fra-eng.zip"
MAX_PAIRS = 20000

def load_pairs(url=DATA_URL, max_pairs=MAX_PAIRS, timeout=15):
    try:
        resp = urllib.request.urlopen(url, timeout=timeout)
        raw_bytes = resp.read()
        zf = zipfile.ZipFile(io.BytesIO(raw_bytes))
        raw_text = zf.read("fra.txt").decode("utf-8")
        lines = raw_text.strip().split("\n")
        pairs = []
        for line in lines:
            parts = line.split("\t")
            if len(parts) >= 2:
                pairs.append((parts[0], parts[1]))
        random.Random(SEED).shuffle(pairs)
        pairs = pairs[:max_pairs]
        print(f"Telechargement reussi : {len(pairs)} paires anglais-francais utilisees.")
        return pairs, True
    except Exception as e:
        print(f"Telechargement impossible ({e!r}).")
        print("-> Repli sur le corpus embarque (resultats indicatifs seulement).")
        return FALLBACK_PAIRS, False

raw_pairs, used_real_data = load_pairs()
print("Exemple de paires :")
for eng, fra in raw_pairs[:5]:
    print(f"  EN: {eng:40s} FR: {fra}")


In [ ]:
# 2.3 Normalisation et tokenisation
def normalize(text):
    text = text.lower().strip()
    text = re.sub(r"([.!?,])", r" \1 ", text)
    text = re.sub(r"[^a-zàâäçéèêëîïôöûùüÿñæœ.!?,' ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return normalize(text).split()

MAX_LEN = 12  # longueur max en tokens (hors <sos>/<eos>) -- garde l'entrainement tractable

eng_tok, fra_tok = [], []
for eng, fra in raw_pairs:
    e_t, f_t = tokenize(eng), tokenize(fra)
    if 1 <= len(e_t) <= MAX_LEN and 1 <= len(f_t) <= MAX_LEN:
        eng_tok.append(e_t)
        fra_tok.append(f_t)

print(f"Paires retenues apres filtrage (longueur <= {MAX_LEN}) : {len(eng_tok)}")


In [ ]:
# 2.4 Construction du vocabulaire (tokens speciaux + seuil de frequence minimale)
class Vocab:
    def __init__(self):
        self.word2idx = {PAD: PAD_IDX, SOS: SOS_IDX, EOS: EOS_IDX, UNK: UNK_IDX}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.counts = Counter()

    def add_sentence(self, tokens):
        self.counts.update(tokens)

    def build(self, min_freq=2):
        for word, c in self.counts.most_common():
            if c >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, tokens, add_sos_eos=True):
        ids = [self.word2idx.get(w, UNK_IDX) for w in tokens]
        if add_sos_eos:
            ids = [SOS_IDX] + ids + [EOS_IDX]
        return ids

    def decode(self, ids, strip_special=True):
        words = [self.idx2word.get(i, UNK) for i in ids]
        if strip_special:
            words = [w for w in words if w not in (PAD, SOS, EOS)]
        return words

    def __len__(self):
        return len(self.word2idx)

eng_vocab = Vocab()
fra_vocab = Vocab()
for toks in eng_tok:
    eng_vocab.add_sentence(toks)
for toks in fra_tok:
    fra_vocab.add_sentence(toks)

eng_vocab.build(min_freq=2)
fra_vocab.build(min_freq=2)

print(f"Taille vocabulaire anglais : {len(eng_vocab)}")
print(f"Taille vocabulaire francais : {len(fra_vocab)}")


In [ ]:
# 2.5 Datasets PyTorch + collate_fn avec padding dynamique (mini-lots de longueurs variables)
class LMDataset(Dataset):
    """Pour le modele de langage : une sequence encodee <sos> w1 w2 ... wn <eos>."""
    def __init__(self, tokenized_sentences, vocab):
        self.data = [torch.tensor(vocab.encode(toks), dtype=torch.long) for toks in tokenized_sentences]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def lm_collate_fn(batch):
    """Pad les sequences ; renvoie (input, target) pour la prediction du prochain token.
    input = seq[:-1], target = seq[1:] (le decalage classique pour un LM autoregressif)."""
    padded = pad_sequence(batch, batch_first=True, padding_value=PAD_IDX)
    return padded[:, :-1], padded[:, 1:]


class Seq2SeqDataset(Dataset):
    """Pour la traduction : paires (source encodee, cible encodee)."""
    def __init__(self, src_tok, tgt_tok, src_vocab, tgt_vocab):
        self.src = [torch.tensor(src_vocab.encode(t), dtype=torch.long) for t in src_tok]
        self.tgt = [torch.tensor(tgt_vocab.encode(t), dtype=torch.long) for t in tgt_tok]

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]


def seq2seq_collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_padded = pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


# Split train / val / test (80/10/10) -- partage entre LM (anglais) et Seq2Seq (fr->en)
n_total = len(eng_tok)
idx_all = list(range(n_total))
random.Random(SEED).shuffle(idx_all)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
train_idx = idx_all[:n_train]
val_idx = idx_all[n_train:n_train + n_val]
test_idx = idx_all[n_train + n_val:]

print(f"Train : {len(train_idx)} | Val : {len(val_idx)} | Test : {len(test_idx)}")

eng_train = [eng_tok[i] for i in train_idx]
eng_val = [eng_tok[i] for i in val_idx]
eng_test = [eng_tok[i] for i in test_idx]

fra_train = [fra_tok[i] for i in train_idx]
fra_val = [fra_tok[i] for i in val_idx]
fra_test = [fra_tok[i] for i in test_idx]

BATCH_SIZE = 64

lm_train_loader = DataLoader(LMDataset(eng_train, eng_vocab), batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=lm_collate_fn)
lm_val_loader = DataLoader(LMDataset(eng_val, eng_vocab), batch_size=BATCH_SIZE,
                            shuffle=False, collate_fn=lm_collate_fn)

xb, yb = next(iter(lm_train_loader))
print("LM batch -- input :", xb.shape, "| target :", yb.shape)


## 3. RNN, LSTM, GRU — rappel des équations

**RNN simple** (cellule de base) :
$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$$
Problème : sur de longues séquences, les gradients rétropropagés à travers $h_{t-1}$ peuvent
s'évanouir (produit répété de termes < 1) ou exploser (produit répété de termes > 1).

**LSTM** introduit des *portes* et une *mémoire cellulaire* $c_t$ séparée de l'état caché :
$$f_t = \sigma(W_f[x_t,h_{t-1}]+b_f) \quad i_t = \sigma(W_i[x_t,h_{t-1}]+b_i) \quad
o_t = \sigma(W_o[x_t,h_{t-1}]+b_o)$$
$$\tilde{c}_t = \tanh(W_c[x_t,h_{t-1}]+b_c) \quad c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
\quad h_t = o_t \odot \tanh(c_t)$$
La porte d'oubli $f_t$ proche de 1 permet à $c_t$ de transporter l'information sur de longues
distances sans atténuation systématique (chemin de gradient quasi-direct).

**GRU** simplifie le LSTM en fusionnant état caché et mémoire cellulaire, avec 2 portes
(reset $r_t$, update $z_t$) au lieu de 3 — moins de paramètres, souvent des performances
comparables au LSTM, plus rapide à entraîner.

In [ ]:
# 3.1 Modele de langage generique (RNN / LSTM / GRU selectionnable)
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, rnn_type="rnn", pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        rnn_classes = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.rnn = rnn_classes[rnn_type](emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.rnn_type = rnn_type

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(out)
        return logits, hidden


In [ ]:
# 3.2 Boucle d'entrainement generique pour le LM + calcul de perplexite
def train_lm(model, train_loader, val_loader, n_epochs=10, lr=1e-3, clip_grad=None, track_grad_norms=False):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": []}
    grad_norms = []

    start = time.time()
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_tokens = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits, _ = model(xb)
            loss = criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            loss.backward()

            if track_grad_norms:
                total_norm = sum(p.grad.data.norm(2).item() ** 2
                                  for p in model.parameters() if p.grad is not None) ** 0.5
                grad_norms.append(total_norm)

            if clip_grad is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)

            optimizer.step()

            n_valid = (yb != PAD_IDX).sum().item()
            total_loss += loss.item() * n_valid
            n_tokens += n_valid

        train_loss = total_loss / n_tokens
        val_loss = evaluate_lm(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_ppl"].append(math.exp(min(train_loss, 20)))
        history["val_ppl"].append(math.exp(min(val_loss, 20)))

        if epoch % 2 == 0 or epoch == 1:
            print(f"  epoch {epoch:2d} | train_loss={train_loss:.3f} (ppl={history['train_ppl'][-1]:.1f}) "
                  f"| val_loss={val_loss:.3f} (ppl={history['val_ppl'][-1]:.1f})")

    elapsed = time.time() - start
    return history, elapsed, grad_norms


def evaluate_lm(model, loader, criterion):
    model.eval()
    total_loss, n_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, _ = model(xb)
            loss = criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            n_valid = (yb != PAD_IDX).sum().item()
            total_loss += loss.item() * n_valid
            n_tokens += n_valid
    return total_loss / n_tokens


## 4. Entraînement comparatif : RNN simple vs LSTM vs GRU

Même architecture (mêmes dimensions d'embedding/hidden), seule la cellule récurrente change.
On compare la perplexité finale, la stabilité de la convergence, le nombre de paramètres et
le temps d'entraînement.

In [ ]:
vocab_size_eng = len(eng_vocab)
lm_results = {}

for rnn_type in ["rnn", "lstm", "gru"]:
    print(f"=== Entrainement LM -- cellule : {rnn_type.upper()} ===")
    torch.manual_seed(SEED)
    model = RNNLanguageModel(vocab_size_eng, emb_dim=128, hidden_dim=256, rnn_type=rnn_type).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    history, elapsed, _ = train_lm(model, lm_train_loader, lm_val_loader, n_epochs=10, lr=1e-3)
    lm_results[rnn_type] = {"model": model, "history": history, "n_params": n_params, "time_s": elapsed}
    print(f"  -> {n_params} parametres, {elapsed:.1f}s, ppl finale (val) = {history['val_ppl'][-1]:.1f}\n")


In [ ]:
# 4.1 Comparaison visuelle de la convergence (perplexite de validation)
fig, ax = plt.subplots(figsize=(8, 5))
for rnn_type, res in lm_results.items():
    ax.plot(res["history"]["val_ppl"], label=rnn_type.upper())
ax.set_xlabel("Epoch")
ax.set_ylabel("Perplexite (validation)")
ax.set_title("RNN vs LSTM vs GRU -- modele de langage")
ax.legend()
plt.show()

summary_lm = pd.DataFrame([
    {"cellule": rnn_type.upper(), "n_params": res["n_params"], "temps_s": round(res["time_s"], 1),
     "ppl_val_finale": round(res["history"]["val_ppl"][-1], 2)}
    for rnn_type, res in lm_results.items()
])
summary_lm


**Lecture attendue** *(à confirmer avec tes résultats réels)* : le RNN simple converge
généralement moins vite et de façon plus instable (oscillations de la perplexité de
validation) que LSTM/GRU, surtout si on augmentait la longueur des séquences. GRU a moins de
paramètres que LSTM pour un hidden_dim identique (2 portes vs 3) et s'entraîne généralement
un peu plus vite, pour une perplexité finale très proche de celle du LSTM sur ce corpus de
taille modeste.

## 5. BPTT et effet du gradient clipping

**BPTT (Backpropagation Through Time)** : on "déroule" le RNN sur les $T$ pas de temps de
la séquence, ce qui revient à un réseau feedforward profond de profondeur $T$ où les poids
sont *partagés* entre les pas de temps. Le gradient total par rapport à un poids partagé est
la somme des gradients à chaque pas de temps. Plus $T$ est grand, plus ce produit de
dérivées successives peut faire **exploser** ou **s'évanouir** le gradient — particulièrement
problématique pour un RNN simple, dont l'absence de portes ne protège pas le signal de
gradient.

**Gradient clipping** : on renormalise le gradient global si sa norme dépasse un seuil
$\theta$ : si $\|g\| > \theta$, alors $g \leftarrow g \cdot \theta / \|g\|$. Cela ne change
pas la *direction* du gradient, seulement sa magnitude — un correctif simple et efficace
contre l'explosion du gradient, sans rien faire contre son évanouissement (pour cela, LSTM/GRU
restent nécessaires).

In [ ]:
# 5.1 Demonstration empirique : RNN simple, AVEC vs SANS gradient clipping
torch.manual_seed(SEED)
model_no_clip = RNNLanguageModel(vocab_size_eng, emb_dim=128, hidden_dim=256, rnn_type="rnn").to(device)
print("--- RNN simple SANS clipping ---")
_, _, grad_norms_no_clip = train_lm(model_no_clip, lm_train_loader, lm_val_loader,
                                     n_epochs=3, lr=1e-2, clip_grad=None, track_grad_norms=True)

torch.manual_seed(SEED)
model_clip = RNNLanguageModel(vocab_size_eng, emb_dim=128, hidden_dim=256, rnn_type="rnn").to(device)
print("\n--- RNN simple AVEC clipping (max_norm=5.0) ---")
_, _, grad_norms_clip = train_lm(model_clip, lm_train_loader, lm_val_loader,
                                  n_epochs=3, lr=1e-2, clip_grad=5.0, track_grad_norms=True)


In [ ]:
# 5.2 Visualisation des normes de gradient par batch -- pic d'instabilite visible sans clipping
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grad_norms_no_clip, label="sans clipping", alpha=0.8)
ax.plot(grad_norms_clip, label="avec clipping (max_norm=5.0)", alpha=0.8)
ax.set_xlabel("Batch (cumule sur les epochs)")
ax.set_ylabel("Norme L2 du gradient global")
ax.set_yscale("log")
ax.set_title("Effet du gradient clipping sur la stabilite de l'entrainement")
ax.legend()
plt.show()

print(f"Norme de gradient max SANS clipping : {max(grad_norms_no_clip):.1f}")
print(f"Norme de gradient max AVEC clipping  : {max(grad_norms_clip):.1f}")


## 6. Système Seq2Seq encodeur–décodeur (traduction français → anglais)

- **Encodeur** : lit la phrase source token par token, condense toute l'information dans
  son dernier état caché $h_T^{enc}$ (le "contexte").
- **Décodeur** : modèle de langage *conditionné* — initialisé avec $h_T^{enc}$, il génère la
  phrase cible token par token, chaque prédiction dépendant de l'état caché courant et du
  token précédent.
- **Teacher forcing** : pendant l'entraînement, on a le choix à chaque pas de temps entre
  utiliser la **vraie** valeur précédente de la cible (teacher forcing) ou la **prédiction**
  du modèle lui-même. Le premier mode accélère et stabilise l'apprentissage (le modèle ne
  s'égare pas sur une séquence d'erreurs en cascade), mais crée un écart train/inférence
  (en inférence, on n'a jamais la vraie cible) — d'où l'usage d'un `teacher_forcing_ratio`
  probabiliste (mélange des deux modes) plutôt que 100% de l'un ou l'autre.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        emb = self.embedding(src)
        _, hidden = self.rnn(emb)
        return hidden  # (1, batch, hidden_dim)


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_step, hidden):
        # input_step : (batch, 1)
        emb = self.embedding(input_step)
        output, hidden = self.rnn(emb, hidden)
        logits = self.fc(output.squeeze(1))
        return logits, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_vocab_size, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        hidden = self.encoder(src)
        input_step = tgt[:, 0].unsqueeze(1)  # <sos>
        outputs = torch.zeros(batch_size, tgt_len, self.tgt_vocab_size, device=self.device)

        for t in range(1, tgt_len):
            logits, hidden = self.decoder(input_step, hidden)
            outputs[:, t] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1)
            input_step = (tgt[:, t] if teacher_force else top1).unsqueeze(1)

        return outputs


In [ ]:
# 6.1 DataLoaders Seq2Seq (francais -> anglais) et boucle d'entrainement avec teacher forcing
seq2seq_train_loader = DataLoader(Seq2SeqDataset(fra_train, eng_train, fra_vocab, eng_vocab),
                                   batch_size=BATCH_SIZE, shuffle=True, collate_fn=seq2seq_collate_fn)
seq2seq_val_loader = DataLoader(Seq2SeqDataset(fra_val, eng_val, fra_vocab, eng_vocab),
                                 batch_size=BATCH_SIZE, shuffle=False, collate_fn=seq2seq_collate_fn)

vocab_size_fra = len(fra_vocab)

def evaluate_seq2seq(model, loader, criterion):
    model.eval()
    total_loss, n_tokens = 0.0, 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            outputs = model(src, tgt, teacher_forcing_ratio=0.0)
            loss = criterion(outputs[:, 1:].reshape(-1, outputs.size(-1)), tgt[:, 1:].reshape(-1))
            n_valid = (tgt[:, 1:] != PAD_IDX).sum().item()
            total_loss += loss.item() * n_valid
            n_tokens += n_valid
    return total_loss / n_tokens


def train_seq2seq(model, train_loader, val_loader, n_epochs=15, lr=1e-3,
                   teacher_forcing_ratio=0.5, clip_grad=1.0):
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_ppl": []}

    start = time.time()
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_tokens = 0.0, 0
        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            outputs = model(src, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
            loss = criterion(outputs[:, 1:].reshape(-1, outputs.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
            optimizer.step()

            n_valid = (tgt[:, 1:] != PAD_IDX).sum().item()
            total_loss += loss.item() * n_valid
            n_tokens += n_valid

        train_loss = total_loss / n_tokens
        val_loss = evaluate_seq2seq(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_ppl"].append(math.exp(min(val_loss, 20)))

        if epoch % 2 == 0 or epoch == 1:
            print(f"  epoch {epoch:2d} | train_loss={train_loss:.3f} | val_loss={val_loss:.3f} "
                  f"| val_ppl={history['val_ppl'][-1]:.1f}")

    elapsed = time.time() - start
    return history, elapsed


torch.manual_seed(SEED)
encoder = Encoder(vocab_size_fra, emb_dim=128, hidden_dim=256).to(device)
decoder = Decoder(vocab_size_eng, emb_dim=128, hidden_dim=256).to(device)
seq2seq_model = Seq2Seq(encoder, decoder, vocab_size_eng, device).to(device)

print("Parametres Seq2Seq :", sum(p.numel() for p in seq2seq_model.parameters()))
seq2seq_history, seq2seq_time = train_seq2seq(seq2seq_model, seq2seq_train_loader, seq2seq_val_loader,
                                               n_epochs=15, lr=1e-3, teacher_forcing_ratio=0.5)


In [ ]:
fig, ax = plt.subplots()
ax.plot(seq2seq_history["train_loss"], label="train")
ax.plot(seq2seq_history["val_loss"], label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Seq2Seq (francais -> anglais) -- courbe d'apprentissage")
ax.legend()
plt.show()
print(f"Temps d'entrainement total : {seq2seq_time:.1f}s")
print(f"Perplexite finale (validation) : {seq2seq_history['val_ppl'][-1]:.2f}")


## 7. Stratégies de décodage : glouton vs beam search

- **Décodage glouton** : à chaque pas, on choisit le token le plus probable selon le
  décodeur, sans jamais revenir en arrière. Rapide, mais une erreur locale (un choix
  sous-optimal à un pas $t$) peut irrémédiablement dégrader toute la suite de la phrase.
- **Beam search** (largeur $k$) : on garde à chaque pas les $k$ séquences partielles les
  plus probables (au lieu d'une seule), et on développe chacune. À la fin, on choisit la
  séquence complète de plus haute probabilité cumulée (log-probabilité, pour éviter les
  soucis numériques de produits de petites probabilités). Plus coûteux ($k\times$ plus de
  calculs par pas) mais explore un espace de recherche plus large, réduisant le risque de
  rester bloqué sur un choix gourmand sous-optimal.

In [ ]:
def greedy_decode(model, src_ids, max_len, device):
    model.eval()
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        hidden = model.encoder(src)
        input_step = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        result = []
        for _ in range(max_len):
            logits, hidden = model.decoder(input_step, hidden)
            top1 = logits.argmax(dim=1).item()
            if top1 == EOS_IDX:
                break
            result.append(top1)
            input_step = torch.tensor([[top1]], dtype=torch.long).to(device)
    return result


def beam_search_decode(model, src_ids, max_len, device, beam_width=3):
    model.eval()
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        hidden0 = model.encoder(src)
        beams = [(0.0, [SOS_IDX], hidden0)]
        completed = []

        for _ in range(max_len):
            candidates = []
            for score, seq, hidden in beams:
                if seq[-1] == EOS_IDX:
                    completed.append((score, seq))
                    continue
                input_step = torch.tensor([[seq[-1]]], dtype=torch.long).to(device)
                logits, hidden_new = model.decoder(input_step, hidden)
                log_probs = F.log_softmax(logits, dim=1).squeeze(0)
                topk_log_probs, topk_idx = log_probs.topk(beam_width)
                for lp, idx in zip(topk_log_probs.tolist(), topk_idx.tolist()):
                    candidates.append((score + lp, seq + [idx], hidden_new))

            if not candidates:
                break
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]
            if all(b[1][-1] == EOS_IDX for b in beams):
                completed.extend([(s, seq) for s, seq, _ in beams])
                break

        if not completed:
            completed = [(s, seq) for s, seq, _ in beams]
        completed.sort(key=lambda x: x[0], reverse=True)
        best_seq = [tok for tok in completed[0][1] if tok not in (SOS_IDX, EOS_IDX)]
    return best_seq


## 8. Évaluation : perplexité et BLEU (glouton vs beam search)

On implémente une version simplifiée mais standard du **BLEU corpus-level** (precision
n-gramme 1 à 4, moyenne géométrique pondérée, pénalité de brièveté) pour ne dépendre d'aucun
package externe supplémentaire.

In [ ]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(references, candidates, max_n=4):
    """references, candidates : listes de listes de tokens (une ref par candidat)."""
    weights = [1.0 / max_n] * max_n
    precisions = []
    for n in range(1, max_n + 1):
        match, total = 0, 0
        for ref, cand in zip(references, candidates):
            ref_counts = ngram_counts(ref, n)
            cand_counts = ngram_counts(cand, n)
            for ng, cnt in cand_counts.items():
                match += min(cnt, ref_counts.get(ng, 0))
            total += max(len(cand) - n + 1, 0)
        precisions.append(match / total if total > 0 else 1e-9)

    precisions = [p if p > 0 else 1e-9 for p in precisions]
    log_avg = sum(w * math.log(p) for w, p in zip(weights, precisions))
    geo_mean = math.exp(log_avg)

    ref_len = sum(len(r) for r in references)
    cand_len = sum(len(c) for c in candidates)
    if cand_len == 0:
        bp = 0.0
    elif cand_len > ref_len:
        bp = 1.0
    else:
        bp = math.exp(1 - ref_len / cand_len)

    return bp * geo_mean


In [ ]:
# 8.1 Evaluation sur le set de test : decodage glouton vs beam search, BLEU compare
MAX_DECODE_LEN = MAX_LEN + 2
N_EVAL = min(200, len(fra_test))  # sous-echantillon pour limiter le temps de calcul

references, greedy_candidates, beam_candidates = [], [], []

for i in range(N_EVAL):
    src_ids = fra_vocab.encode(fra_test[i])
    ref_tokens = eng_test[i]

    greedy_ids = greedy_decode(seq2seq_model, src_ids, MAX_DECODE_LEN, device)
    beam_ids = beam_search_decode(seq2seq_model, src_ids, MAX_DECODE_LEN, device, beam_width=3)

    references.append(ref_tokens)
    greedy_candidates.append(eng_vocab.decode(greedy_ids, strip_special=False))
    beam_candidates.append(eng_vocab.decode(beam_ids, strip_special=False))

bleu_greedy = corpus_bleu(references, greedy_candidates)
bleu_beam = corpus_bleu(references, beam_candidates)

print(f"BLEU (decodage glouton) : {bleu_greedy:.4f}")
print(f"BLEU (beam search, k=3) : {bleu_beam:.4f}")


In [ ]:
# 8.2 Exemples qualitatifs de traduction
print(f"{'Source (FR)':35s} | {'Reference (EN)':30s} | {'Glouton':30s} | {'Beam search'}")
print("-" * 130)
for i in range(8):
    src_text = " ".join(fra_test[i])
    ref_text = " ".join(references[i])
    greedy_text = " ".join(greedy_candidates[i])
    beam_text = " ".join(beam_candidates[i])
    print(f"{src_text[:33]:35s} | {ref_text[:28]:30s} | {greedy_text[:28]:30s} | {beam_text}")


## 9. Analyse critique *(template — à compléter avec tes résultats réels)*

- **Stabilité / performance / mémoire / coût de calcul (section 4)** : tableau comparatif
  RNN/LSTM/GRU — citer les chiffres exacts obtenus (`summary_lm`), commenter lequel converge
  le plus vite et le plus stablement, et à quel coût en paramètres/temps.
- **BPTT et gradient clipping (section 5)** : citer la norme de gradient maximale observée
  avec et sans clipping — si l'écart est important, c'est une preuve empirique directe de
  l'instabilité du RNN simple sur séquences même courtes (12 tokens ici).
- **Teacher forcing** : ce notebook utilise un ratio fixe de 0.5 ; une amélioration classique
  serait un *scheduled sampling* (ratio qui diminue avec les epochs) pour réduire l'écart
  train/inférence — piste d'amélioration à mentionner même si non implémentée.
- **Glouton vs beam search (section 8)** : comparer `bleu_greedy` et `bleu_beam` — le gain du
  beam search est généralement modeste sur des phrases courtes et un corpus de taille
  modeste, mais devient plus net sur des séquences plus longues où les erreurs gloutonnes
  se cumulent davantage.
- **Limites** : corpus limité à des phrases de ≤ 12 tokens (pour la tractabilité), vocabulaire
  restreint par le seuil `min_freq=2` (les mots rares deviennent `<unk>`), pas de mécanisme
  d'attention (la dépendance à un unique vecteur de contexte de taille fixe limite la
  qualité sur des phrases plus longues — c'est précisément la limite qui a motivé
  l'invention de l'attention/des Transformers).

## 10. Question de synthèse — Partie III *(template à compléter)*

> *Dans quelle mesure les architectures récurrentes permettent-elles de modéliser
> efficacement une séquence réelle, et comment justifier le passage d'un RNN simple vers
> un LSTM/GRU puis vers un schéma encodeur–décodeur pour une tâche de génération ou de
> traduction ?*

**Plan de réponse à rédiger :**
1. **Modélisation probabiliste** — la factorisation par la règle de chaîne (section 1)
   justifie l'approche autoregressive commune au LM et au décodeur du Seq2Seq.
2. **Mémoire** — les portes du LSTM/GRU (section 3) répondent directement au problème du
   gradient évanescent du RNN simple démontré empiriquement en section 5 ; citer les chiffres
   de stabilité observés.
3. **Entraînement** — teacher forcing (section 6), gradient clipping (section 5), et leur
   rôle dans la stabilité de l'apprentissage sur séquences.
4. **Qualité du décodage** — comparer perplexité (LM, section 4) et BLEU glouton/beam
   (section 8) ; relier la limite de l'unique vecteur de contexte fixe à la nécessité, en
   recherche, de mécanismes d'attention pour des séquences plus longues.
5. **Limites observées expérimentalement** — taille du corpus, longueur maximale des
   séquences, absence d'attention — à formuler comme perspectives d'amélioration plutôt que
   comme défauts du raisonnement théorique.